# การจำแนกประเภทระยะการนอนหลับ (Advanced Version)

## ภาพรวม
โน้ตบุ๊กนี้ใช้วิธีการขั้นสูง:
1. **การสกัดคุณลักษณะที่รวมย่อย**: สถิติโดเมนเวลา + ความถี่ + HRV + ความสัมพันธ์ข้ามช่องสัญญาณ
2. **บริบทเวลา**: ค่าเพื่อนบ้าน (±1 ถึง ±3) + เดลต้า + ค่าเฉลี่ยเพื่อนบ้าน
3. **Ensemble**: 4 LightGBM + 1 XGBoost กับ GroupKFold (5 folds)
4. **Post-processing**: Median filter + Viterbi (HMM) smoothing พร้อมการเลือกอัตโนมัติ

In [ ]:
# ติดตั้ง dependencies
!pip install kaggle lightgbm xgboost scipy -q

In [ ]:
# Kaggle API setup และ download data
import os
from pathlib import Path

# ตั้งค่า Kaggle credentials
os.environ['KAGGLE_CONFIG_DIR'] = '/root/.kaggle'
if not Path('/root/.kaggle/kaggle.json').exists():
    # สำหรับ Colab: upload kaggle.json หรือสร้างมันเอง
    print("กรุณาตั้งค่า Kaggle API credentials")

# Download competition data
!kaggle competitions download -c super-ai-engineer-ss-6-sleep-stage-classification -p /tmp/sleep_data --force
!unzip -q /tmp/sleep_data/super-ai-engineer-ss-6-sleep-stage-classification.zip -d /tmp/sleep_data

DATA_DIR = Path('/tmp/sleep_data')
print(f"Data directory: {DATA_DIR}")
print(f"Contents: {list(DATA_DIR.iterdir())}")

In [ ]:
# Imports และ constants
import pandas as pd
import numpy as np
from pathlib import Path
from typing import Dict, Tuple, List, Any, Optional
from dataclasses import dataclass
from scipy import signal, stats
from scipy.signal import welch
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import GroupKFold
from sklearn.preprocessing import LabelEncoder
import lightgbm as lgb
import xgboost as xgb
from sklearn.metrics import accuracy_score, f1_score

# Constants
FEATURE_COLS = ['BVP', 'ACC_X', 'ACC_Y', 'ACC_Z', 'TEMP', 'EDA', 'HR', 'IBI']
SEGMENT_SIZE = 480
FS = 16.0
N_CLASSES = 5
LABEL_ENCODER = {'W': 0, 'N1': 1, 'N2': 2, 'N3': 3, 'R': 4}
LABEL_DECODER = {v: k for k, v in LABEL_ENCODER.items()}

print(f"Constants defined:")
print(f"  SEGMENT_SIZE={SEGMENT_SIZE}, FS={FS}, N_CLASSES={N_CLASSES}")
print(f"  Features: {FEATURE_COLS}")

In [ ]:
# Utility functions

def _sanitize(x: Any) -> float:
    """แปลงค่า NaN/inf เป็น 0"""
    if isinstance(x, (list, np.ndarray)):
        x = float(np.nanmean(x)) if len(x) > 0 else 0.0
    if pd.isna(x) or np.isinf(x):
        return 0.0
    return float(x)

def _safe_corr(x: np.ndarray, y: np.ndarray) -> float:
    """คำนวณความสัมพันธ์อย่างปลอดภัย"""
    x = np.asarray(x).flatten()
    y = np.asarray(y).flatten()
    if len(x) < 2 or len(y) < 2 or np.std(x) == 0 or np.std(y) == 0:
        return 0.0
    try:
        return float(np.corrcoef(x, y)[0, 1])
    except:
        return 0.0

def _dominant_label(labels: List[str]) -> str:
    """หาป้ายกำกับที่พบบ่อยที่สุด"""
    if not labels:
        return 'N2'
    counts = {}
    for l in labels:
        counts[l] = counts.get(l, 0) + 1
    return max(counts.items(), key=lambda x: x[1])[0]

def log(msg: str):
    """พิมพ์ข้อความ"""
    print(f"[LOG] {msg}")

print("Utility functions defined")

## การสกัดคุณลักษณะ (Feature Extraction)

In [ ]:
def _channel_stats(channel_data: np.ndarray) -> Dict[str, float]:
    """
    สถิติโดเมนเวลาสำหรับช่องเดียว
    ประกอบด้วย: mean, std, min, max, median, range, q10, q25, q75, q90, iqr,
    energy, rms, mad, skew, kurtosis, entropy, nan_ratio, diff_mean, diff_std, diff_max,
    zero_crossings, autocorr_lag1, trend_slope
    """
    x = np.asarray(channel_data).flatten()
    x_clean = x[~np.isnan(x)]
    
    if len(x_clean) == 0:
        return {f"{name}": 0.0 for name in [
            'mean', 'std', 'min', 'max', 'median', 'range', 'q10', 'q25', 'q75', 'q90', 'iqr',
            'energy', 'rms', 'mad', 'skew', 'kurtosis', 'entropy', 'nan_ratio',
            'diff_mean', 'diff_std', 'diff_max', 'zero_crossings', 'autocorr_lag1', 'trend_slope'
        ]}
    
    stats_dict = {}
    
    # สถิติพื้นฐาน
    stats_dict['mean'] = _sanitize(np.mean(x_clean))
    stats_dict['std'] = _sanitize(np.std(x_clean))
    stats_dict['min'] = _sanitize(np.min(x_clean))
    stats_dict['max'] = _sanitize(np.max(x_clean))
    stats_dict['median'] = _sanitize(np.median(x_clean))
    stats_dict['range'] = _sanitize(stats_dict['max'] - stats_dict['min'])
    
    # Percentiles
    q10, q25, q75, q90 = np.percentile(x_clean, [10, 25, 75, 90])
    stats_dict['q10'] = _sanitize(q10)
    stats_dict['q25'] = _sanitize(q25)
    stats_dict['q75'] = _sanitize(q75)
    stats_dict['q90'] = _sanitize(q90)
    stats_dict['iqr'] = _sanitize(q75 - q25)
    
    # Energy and RMS
    stats_dict['energy'] = _sanitize(np.sum(x_clean ** 2))
    stats_dict['rms'] = _sanitize(np.sqrt(np.mean(x_clean ** 2)))
    
    # MAD (Mean Absolute Deviation)
    stats_dict['mad'] = _sanitize(np.mean(np.abs(x_clean - np.mean(x_clean))))
    
    # Skewness and Kurtosis
    stats_dict['skew'] = _sanitize(stats.skew(x_clean))
    stats_dict['kurtosis'] = _sanitize(stats.kurtosis(x_clean))
    
    # Entropy (ฮิสโตแกรม)
    hist, _ = np.histogram(x_clean, bins=min(30, len(x_clean)//10 + 1))
    hist = hist[hist > 0]
    prob = hist / np.sum(hist)
    stats_dict['entropy'] = _sanitize(-np.sum(prob * np.log2(prob + 1e-10)))
    
    # NaN ratio
    stats_dict['nan_ratio'] = _sanitize(np.sum(np.isnan(x)) / len(x))
    
    # Differences (derivatives)
    if len(x_clean) > 1:
        diff = np.diff(x_clean)
        stats_dict['diff_mean'] = _sanitize(np.mean(diff))
        stats_dict['diff_std'] = _sanitize(np.std(diff))
        stats_dict['diff_max'] = _sanitize(np.max(np.abs(diff)))
    else:
        stats_dict['diff_mean'] = 0.0
        stats_dict['diff_std'] = 0.0
        stats_dict['diff_max'] = 0.0
    
    # Zero crossings
    if len(x_clean) > 1:
        zero_crossings = np.sum(np.diff(np.sign(x_clean - np.mean(x_clean))) != 0)
        stats_dict['zero_crossings'] = _sanitize(zero_crossings)
    else:
        stats_dict['zero_crossings'] = 0.0
    
    # Autocorrelation at lag 1
    if len(x_clean) > 1:
        x_centered = x_clean - np.mean(x_clean)
        autocorr = np.correlate(x_centered, x_centered, mode='full')
        autocorr = autocorr[len(autocorr)//2:]
        if autocorr[0] > 0 and len(autocorr) > 1:
            stats_dict['autocorr_lag1'] = _sanitize(autocorr[1] / autocorr[0])
        else:
            stats_dict['autocorr_lag1'] = 0.0
    else:
        stats_dict['autocorr_lag1'] = 0.0
    
    # Trend slope (linear regression)
    if len(x_clean) > 1:
        x_idx = np.arange(len(x_clean))
        z = np.polyfit(x_idx, x_clean, 1)
        stats_dict['trend_slope'] = _sanitize(z[0])
    else:
        stats_dict['trend_slope'] = 0.0
    
    return stats_dict

print("_channel_stats function defined")

In [ ]:
def _channel_freq_features(channel_data: np.ndarray, fs: float = FS) -> Dict[str, float]:
    """
    คุณลักษณะโดเมนความถี่ใช้ Welch PSD
    ประกอบด้วย: ultra_low, delta, theta, alpha, beta (band powers),
    dominant_freq, spectral_entropy, spectral_centroid, spectral_bandwidth,
    spectral_rolloff85, lf_hf_ratio
    """
    x = np.asarray(channel_data).flatten()
    x = x[~np.isnan(x)]
    
    freq_dict = {}
    
    # Default return ถ้ามีข้อมูลน้อยเกินไป
    if len(x) < 10:
        default_keys = ['ultra_low', 'delta', 'theta', 'alpha', 'beta',
                       'dominant_freq', 'spectral_entropy', 'spectral_centroid',
                       'spectral_bandwidth', 'spectral_rolloff85', 'lf_hf_ratio']
        return {k: 0.0 for k in default_keys}
    
    try:
        # Welch PSD
        freqs, psd = welch(x, fs=fs, nperseg=min(len(x), 256))
        
        # Band definitions (Hz)
        bands = {
            'ultra_low': (0, 0.1),
            'delta': (0.1, 4),
            'theta': (4, 8),
            'alpha': (8, 12),
            'beta': (12, 30)
        }
        
        # Band powers
        for band_name, (low, high) in bands.items():
            mask = (freqs >= low) & (freqs < high)
            if np.any(mask):
                freq_dict[band_name] = _sanitize(np.mean(psd[mask]))
            else:
                freq_dict[band_name] = 0.0
        
        # Dominant frequency
        idx = np.argmax(psd)
        freq_dict['dominant_freq'] = _sanitize(freqs[idx] if idx < len(freqs) else 0.0)
        
        # Spectral entropy
        psd_norm = psd / (np.sum(psd) + 1e-10)
        freq_dict['spectral_entropy'] = _sanitize(-np.sum(psd_norm * np.log2(psd_norm + 1e-10)))
        
        # Spectral centroid
        freq_dict['spectral_centroid'] = _sanitize(np.sum(freqs * psd) / (np.sum(psd) + 1e-10))
        
        # Spectral bandwidth
        centroid = freq_dict['spectral_centroid']
        numerator = np.sum(((freqs - centroid) ** 2) * psd)
        denominator = np.sum(psd) + 1e-10
        freq_dict['spectral_bandwidth'] = _sanitize(np.sqrt(numerator / denominator))
        
        # Spectral rolloff (85th percentile)
        psd_cumsum = np.cumsum(psd)
        rolloff_idx = np.where(psd_cumsum >= 0.85 * psd_cumsum[-1])[0]
        if len(rolloff_idx) > 0:
            freq_dict['spectral_rolloff85'] = _sanitize(freqs[rolloff_idx[0]])
        else:
            freq_dict['spectral_rolloff85'] = _sanitize(freqs[-1])
        
        # LF/HF ratio (LF: 0.1-4 Hz, HF: 4-12 Hz)
        lf_mask = (freqs >= 0.1) & (freqs < 4)
        hf_mask = (freqs >= 4) & (freqs < 12)
        lf_power = np.mean(psd[lf_mask]) if np.any(lf_mask) else 1e-10
        hf_power = np.mean(psd[hf_mask]) if np.any(hf_mask) else 1e-10
        freq_dict['lf_hf_ratio'] = _sanitize(lf_power / (hf_power + 1e-10))
        
    except Exception as e:
        default_keys = ['ultra_low', 'delta', 'theta', 'alpha', 'beta',
                       'dominant_freq', 'spectral_entropy', 'spectral_centroid',
                       'spectral_bandwidth', 'spectral_rolloff85', 'lf_hf_ratio']
        freq_dict = {k: 0.0 for k in default_keys}
    
    return freq_dict

print("_channel_freq_features function defined")

In [ ]:
def _sub_segment_features(channel_data: np.ndarray) -> Dict[str, float]:
    """
    แบ่งช่องสัญญาณ 30 วินาที (480 samples) เป็น 3 หน้าต่าง 10 วินาที (160 samples)
    คำนวณ mean, std สำหรับแต่ละหน้าต่าง + trend + max_change
    """
    x = np.asarray(channel_data).flatten()
    x = x[~np.isnan(x)]
    
    sub_dict = {}
    
    # แบ่ง 3 ส่วน
    segment_size = len(x) // 3 if len(x) >= 3 else len(x)
    segments = [x[i*segment_size:(i+1)*segment_size] for i in range(3)]
    
    # Mean and std สำหรับแต่ละส่วน
    for i, seg in enumerate(segments, 1):
        if len(seg) > 0:
            sub_dict[f'seg{i}_mean'] = _sanitize(np.mean(seg))
            sub_dict[f'seg{i}_std'] = _sanitize(np.std(seg))
        else:
            sub_dict[f'seg{i}_mean'] = 0.0
            sub_dict[f'seg{i}_std'] = 0.0
    
    # Trend: linear slope ของ 3 ส่วน
    if len(segments) >= 3:
        seg_means = [np.mean(seg) if len(seg) > 0 else 0.0 for seg in segments]
        z = np.polyfit([0, 1, 2], seg_means, 1)
        sub_dict['seg_trend'] = _sanitize(z[0])
    else:
        sub_dict['seg_trend'] = 0.0
    
    # Max change between segments
    if len(segments) >= 2:
        max_change = 0.0
        for i in range(len(segments) - 1):
            m1 = np.mean(segments[i]) if len(segments[i]) > 0 else 0.0
            m2 = np.mean(segments[i+1]) if len(segments[i+1]) > 0 else 0.0
            max_change = max(max_change, abs(m2 - m1))
        sub_dict['seg_max_change'] = _sanitize(max_change)
    else:
        sub_dict['seg_max_change'] = 0.0
    
    return sub_dict

print("_sub_segment_features function defined")

In [ ]:
def _hrv_features_deduped(ibi_data: np.ndarray) -> Dict[str, float]:
    """
    HRV features จาก deduplicated IBI
    ประกอบด้วย: sdnn, rmssd, pnn50, sdsd, mean_ibi, median_ibi, cv_ibi, range_ibi, tri_index
    """
    ibi = np.asarray(ibi_data).flatten()
    ibi = ibi[~np.isnan(ibi)]
    
    hrv_dict = {}
    
    if len(ibi) < 2:
        return {
            'sdnn': 0.0, 'rmssd': 0.0, 'pnn50': 0.0, 'sdsd': 0.0,
            'mean_ibi': 0.0, 'median_ibi': 0.0, 'cv_ibi': 0.0, 'range_ibi': 0.0, 'tri_index': 0.0
        }
    
    # Deduplicate IBI (remove zeros or very small values)
    ibi_dedup = ibi[ibi > 0.1]
    
    if len(ibi_dedup) < 2:
        return {
            'sdnn': 0.0, 'rmssd': 0.0, 'pnn50': 0.0, 'sdsd': 0.0,
            'mean_ibi': 0.0, 'median_ibi': 0.0, 'cv_ibi': 0.0, 'range_ibi': 0.0, 'tri_index': 0.0
        }
    
    # SDNN: std of IBI
    hrv_dict['sdnn'] = _sanitize(np.std(ibi_dedup))
    
    # RMSSD: root mean square of successive differences
    diff_ibi = np.diff(ibi_dedup)
    hrv_dict['rmssd'] = _sanitize(np.sqrt(np.mean(diff_ibi ** 2)))
    
    # pNN50: percentage of successive IBI differences > 50 ms
    if len(diff_ibi) > 0:
        pnn50_count = np.sum(np.abs(diff_ibi) > 0.05)
        hrv_dict['pnn50'] = _sanitize(100.0 * pnn50_count / len(diff_ibi))
    else:
        hrv_dict['pnn50'] = 0.0
    
    # SDSD: std of successive differences
    if len(diff_ibi) > 0:
        hrv_dict['sdsd'] = _sanitize(np.std(diff_ibi))
    else:
        hrv_dict['sdsd'] = 0.0
    
    # Mean and median IBI
    hrv_dict['mean_ibi'] = _sanitize(np.mean(ibi_dedup))
    hrv_dict['median_ibi'] = _sanitize(np.median(ibi_dedup))
    
    # Coefficient of variation
    if hrv_dict['mean_ibi'] > 0:
        hrv_dict['cv_ibi'] = _sanitize(hrv_dict['sdnn'] / hrv_dict['mean_ibi'])
    else:
        hrv_dict['cv_ibi'] = 0.0
    
    # Range
    hrv_dict['range_ibi'] = _sanitize(np.max(ibi_dedup) - np.min(ibi_dedup))
    
    # Triangular index: length / height of histogram
    hist, edges = np.histogram(ibi_dedup, bins=min(100, len(ibi_dedup)//10 + 1))
    if np.max(hist) > 0:
        hrv_dict['tri_index'] = _sanitize(len(ibi_dedup) / np.max(hist))
    else:
        hrv_dict['tri_index'] = 0.0
    
    return hrv_dict

print("_hrv_features_deduped function defined")

In [ ]:
def extract_features(row_dict: Dict[str, np.ndarray]) -> Dict[str, float]:
    """
    สกัดคุณลักษณะทั้งหมด: time-domain + frequency + HRV + sub-segment + correlations + jerk
    row_dict: dictionary with keys = FEATURE_COLS
    """
    features = {}
    
    # 1. Time-domain stats สำหรับแต่ละช่อง
    for col in FEATURE_COLS:
        if col in row_dict:
            stats = _channel_stats(row_dict[col])
            for stat_name, val in stats.items():
                features[f"{col}_{stat_name}"] = val
    
    # 2. Frequency features
    for col in ['BVP', 'EDA', 'HR']:
        if col in row_dict:
            freq_features = _channel_freq_features(row_dict[col])
            for freq_name, val in freq_features.items():
                features[f"{col}_freq_{freq_name}"] = val
    
    # 3. Sub-segment features
    for col in ['BVP', 'ACC_X', 'ACC_Y', 'ACC_Z']:
        if col in row_dict:
            sub_feat = _sub_segment_features(row_dict[col])
            for sub_name, val in sub_feat.items():
                features[f"{col}_subseg_{sub_name}"] = val
    
    # 4. HRV from IBI
    if 'IBI' in row_dict:
        hrv_features = _hrv_features_deduped(row_dict['IBI'])
        for hrv_name, val in hrv_features.items():
            features[f"IBI_hrv_{hrv_name}"] = val
    
    # 5. Accelerometer magnitude
    if 'ACC_X' in row_dict and 'ACC_Y' in row_dict and 'ACC_Z' in row_dict:
        acc_x = np.asarray(row_dict['ACC_X']).flatten()
        acc_y = np.asarray(row_dict['ACC_Y']).flatten()
        acc_z = np.asarray(row_dict['ACC_Z']).flatten()
        
        # Magnitude
        acc_mag = np.sqrt(acc_x**2 + acc_y**2 + acc_z**2)
        acc_mag = acc_mag[~np.isnan(acc_mag)]
        
        if len(acc_mag) > 0:
            features['ACC_mag_mean'] = _sanitize(np.mean(acc_mag))
            features['ACC_mag_std'] = _sanitize(np.std(acc_mag))
            features['ACC_mag_max'] = _sanitize(np.max(acc_mag))
            features['ACC_mag_min'] = _sanitize(np.min(acc_mag))
            features['ACC_mag_energy'] = _sanitize(np.sum(acc_mag ** 2))
            features['ACC_mag_median'] = _sanitize(np.median(acc_mag))
            
            # Movement ratio (percentage > threshold)
            threshold = np.mean(acc_mag) + np.std(acc_mag)
            features['ACC_movement_ratio'] = _sanitize(np.sum(acc_mag > threshold) / len(acc_mag))
            
            # Stillness ratio
            stillness_threshold = np.mean(acc_mag) - np.std(acc_mag)
            features['ACC_stillness_ratio'] = _sanitize(np.sum(acc_mag < max(0, stillness_threshold)) / len(acc_mag))
        else:
            features['ACC_mag_mean'] = 0.0
            features['ACC_mag_std'] = 0.0
            features['ACC_mag_max'] = 0.0
            features['ACC_mag_min'] = 0.0
            features['ACC_mag_energy'] = 0.0
            features['ACC_mag_median'] = 0.0
            features['ACC_movement_ratio'] = 0.0
            features['ACC_stillness_ratio'] = 0.0
        
        # Jerk (อนุพันธ์ของความเร่ง)
        if len(acc_x) > 1 and len(acc_y) > 1 and len(acc_z) > 1:
            acc_x_clean = acc_x[~np.isnan(acc_x)]
            acc_y_clean = acc_y[~np.isnan(acc_y)]
            acc_z_clean = acc_z[~np.isnan(acc_z)]
            
            # Truncate to same length
            min_len = min(len(acc_x_clean), len(acc_y_clean), len(acc_z_clean))
            if min_len > 1:
                acc_x_clean = acc_x_clean[:min_len]
                acc_y_clean = acc_y_clean[:min_len]
                acc_z_clean = acc_z_clean[:min_len]
                
                jerk_x = np.diff(acc_x_clean)
                jerk_y = np.diff(acc_y_clean)
                jerk_z = np.diff(acc_z_clean)
                jerk_mag = np.sqrt(jerk_x**2 + jerk_y**2 + jerk_z**2)
                
                features['jerk_mean'] = _sanitize(np.mean(jerk_mag))
                features['jerk_std'] = _sanitize(np.std(jerk_mag))
                features['jerk_max'] = _sanitize(np.max(jerk_mag))
            else:
                features['jerk_mean'] = 0.0
                features['jerk_std'] = 0.0
                features['jerk_max'] = 0.0
        else:
            features['jerk_mean'] = 0.0
            features['jerk_std'] = 0.0
            features['jerk_max'] = 0.0
    
    # 6. Cross-channel correlations
    corr_pairs = [
        ('BVP', 'HR'), ('BVP', 'EDA'), ('HR', 'EDA'),
        ('ACC_X', 'ACC_Y'), ('ACC_Y', 'ACC_Z'), ('ACC_X', 'ACC_Z')
    ]
    for col1, col2 in corr_pairs:
        if col1 in row_dict and col2 in row_dict:
            corr_val = _safe_corr(row_dict[col1], row_dict[col2])
            features[f"{col1}_{col2}_corr"] = corr_val
    
    # 7. Ratio features
    if 'ACC_X' in row_dict and 'ACC_Y' in row_dict and 'ACC_Z' in row_dict:
        ax = np.asarray(row_dict['ACC_X']).flatten()
        ay = np.asarray(row_dict['ACC_Y']).flatten()
        az = np.asarray(row_dict['ACC_Z']).flatten()
        
        ax_std = np.std(ax[~np.isnan(ax)]) if len(ax[~np.isnan(ax)]) > 0 else 0.1
        ay_std = np.std(ay[~np.isnan(ay)]) if len(ay[~np.isnan(ay)]) > 0 else 0.1
        az_std = np.std(az[~np.isnan(az)]) if len(az[~np.isnan(az)]) > 0 else 0.1
        
        features['ACC_XY_ratio'] = _sanitize(ax_std / (ay_std + 1e-10))
        features['ACC_YZ_ratio'] = _sanitize(ay_std / (az_std + 1e-10))
        features['ACC_XZ_ratio'] = _sanitize(ax_std / (az_std + 1e-10))
    
    return features

print("extract_features function defined")

## การโหลดข้อมูล (Data Loading)

In [ ]:
def load_training_data(train_dir: Path, segment_size: int = SEGMENT_SIZE) -> Tuple[pd.DataFrame, np.ndarray, np.ndarray]:
    """
    โหลดข้อมูลการฝึกอบรม แยก 30 วินาที แยกกลุ่มตามธีม
    ส่งกลับ: (df_features, y_labels, subject_ids)
    """
    train_files = sorted(train_dir.glob('*.csv'))
    all_features = []
    all_labels = []
    all_subjects = []
    
    for file_path in train_files:
        subject_id = file_path.stem.split('_')[0]
        log(f"Loading {file_path.name}")
        
        df = pd.read_csv(file_path, index_col=0)
        df = df[FEATURE_COLS].fillna(0)
        
        n_rows = len(df)
        n_segments = n_rows // segment_size
        
        for seg_idx in range(n_segments):
            start_idx = seg_idx * segment_size
            end_idx = start_idx + segment_size
            
            segment_data = df.iloc[start_idx:end_idx]
            labels = df.iloc[start_idx:end_idx].index.tolist()
            
            # สร้าง row dict สำหรับ extract_features
            row_dict = {col: segment_data[col].values for col in FEATURE_COLS}
            features = extract_features(row_dict)
            
            # เพิ่ม position_in_night (normalized segment index)
            position = seg_idx / max(n_segments, 1)
            features['position_in_night'] = position
            features['position_sin'] = np.sin(2 * np.pi * position)
            features['position_cos'] = np.cos(2 * np.pi * position)
            
            all_features.append(features)
            
            # หา label ที่พบบ่อยที่สุด
            dominant_label = _dominant_label(labels)
            all_labels.append(LABEL_ENCODER.get(dominant_label, 2))
            all_subjects.append(subject_id)
    
    df_features = pd.DataFrame(all_features).fillna(0)
    y_labels = np.array(all_labels)
    subject_ids = np.array(all_subjects)
    
    log(f"Loaded {len(df_features)} segments from {len(train_files)} files")
    return df_features, y_labels, subject_ids

print("load_training_data function defined")

In [ ]:
def load_test_data(test_dir: Path) -> Tuple[Dict[str, pd.DataFrame], Dict[str, List[str]]]:
    """
    โหลดข้อมูลการทดสอบ (segments per subject)
    ส่งกลับ: (subject_test_data, subject_test_ids)
    """
    subject_dirs = [d for d in test_dir.iterdir() if d.is_dir()]
    subject_test_data = {}
    subject_test_ids = {}
    
    for subj_dir in sorted(subject_dirs):
        subject_id = subj_dir.name
        log(f"Loading test data for {subject_id}")
        
        csv_files = sorted(subj_dir.glob('*.csv'))
        test_features = []
        test_ids = []
        
        for file_path in csv_files:
            segment_id = file_path.stem
            df = pd.read_csv(file_path, index_col=0)
            df = df[FEATURE_COLS].fillna(0)
            
            # Extract features from entire segment
            row_dict = {col: df[col].values for col in FEATURE_COLS}
            features = extract_features(row_dict)
            features['position_in_night'] = 0.5  # default position
            features['position_sin'] = np.sin(2 * np.pi * 0.5)
            features['position_cos'] = np.cos(2 * np.pi * 0.5)
            
            test_features.append(features)
            test_ids.append(segment_id)
        
        df_test = pd.DataFrame(test_features).fillna(0)
        subject_test_data[subject_id] = df_test
        subject_test_ids[subject_id] = test_ids
    
    log(f"Loaded test data for {len(subject_test_data)} subjects")
    return subject_test_data, subject_test_ids

print("load_test_data function defined")

## บริบทเวลา (Temporal Context)

In [ ]:
def select_temporal_seed_features(X_train: pd.DataFrame, y_train: np.ndarray, n_features: int = 50) -> List[str]:
    """
    ฝึก LightGBM อย่างรวดเร็ว เลือก top n features ตามความสำคัญ
    """
    log(f"Selecting top {n_features} seed features...")
    
    model = lgb.LGBMClassifier(
        n_estimators=100,
        num_leaves=31,
        learning_rate=0.1,
        random_state=42,
        verbose=-1
    )
    model.fit(X_train, y_train)
    
    importance = pd.Series(
        model.feature_importances_,
        index=X_train.columns
    ).sort_values(ascending=False)
    
    top_features = importance.head(n_features).index.tolist()
    log(f"Selected {len(top_features)} seed features")
    return top_features

def add_temporal_context(
    X: pd.DataFrame,
    seed_features: List[str],
    offsets: List[int] = [1, 2, 3]
) -> pd.DataFrame:
    """
    เพิ่มค่าเพื่อนบ้าน + delta + ค่าเฉลี่ยเพื่อนบ้าน
    """
    X_temporal = X.copy()
    
    for offset in offsets:
        # ค่าเพื่อนบ้านก่อนหน้า
        for feat in seed_features:
            if feat in X.columns:
                X_temporal[f"{feat}_lag_{offset}"] = X[feat].shift(offset).fillna(0)
        
        # ค่าเพื่อนบ้านข้างหน้า
        for feat in seed_features:
            if feat in X.columns:
                X_temporal[f"{feat}_lead_{offset}"] = X[feat].shift(-offset).fillna(0)
    
    # เดลต้า (ความแตกต่าง)
    for feat in seed_features:
        if feat in X.columns:
            X_temporal[f"{feat}_delta1"] = X[feat].diff().fillna(0)
            X_temporal[f"{feat}_delta2"] = X[feat].diff(2).fillna(0)
    
    # ค่าเฉลี่ยของเพื่อนบ้าน (window=3)
    for feat in seed_features:
        if feat in X.columns:
            X_temporal[f"{feat}_neighbor_mean"] = X[feat].rolling(3, center=True).mean().fillna(0)
    
    log(f"Added temporal context: {X_temporal.shape[1]} features")
    return X_temporal

print("Temporal context functions defined")

## Post-Processing (Smoothing)

In [ ]:
def median_filter_smooth(predictions: np.ndarray, window_size: int = 5) -> np.ndarray:
    """
    ใช้ median filter เพื่อปรับให้เรียบ
    """
    from scipy.ndimage import median_filter
    return median_filter(predictions, size=window_size, mode='nearest').astype(int)

def estimate_transition_matrix(y_true: np.ndarray, y_pred: np.ndarray) -> np.ndarray:
    """
    ประมาณเมทริกซ์ transition สำหรับ Viterbi smoothing
    """
    n_states = N_CLASSES
    transition = np.ones((n_states, n_states)) * 0.01  # small default
    
    for i in range(len(y_pred) - 1):
        from_state = int(y_pred[i])
        to_state = int(y_pred[i + 1])
        if 0 <= from_state < n_states and 0 <= to_state < n_states:
            transition[from_state, to_state] += 1
    
    # Normalize
    for i in range(n_states):
        s = np.sum(transition[i, :])
        if s > 0:
            transition[i, :] /= s
    
    return transition

def viterbi_smooth(predictions: np.ndarray, y_train: np.ndarray) -> np.ndarray:
    """
    Viterbi smoothing โดยใช้ HMM
    """
    transition_matrix = estimate_transition_matrix(y_train, y_train)
    
    n_states = N_CLASSES
    n_obs = len(predictions)
    
    # Emission probability (confidence)
    emission = np.zeros((n_obs, n_states))
    for i, pred in enumerate(predictions):
        pred_int = int(pred)
        if 0 <= pred_int < n_states:
            emission[i, pred_int] = 0.9
            other_prob = 0.1 / (n_states - 1)
            for j in range(n_states):
                if j != pred_int:
                    emission[i, j] = other_prob
    
    # Viterbi algorithm
    log_transition = np.log(transition_matrix + 1e-10)
    log_emission = np.log(emission + 1e-10)
    
    viterbi = np.zeros((n_obs, n_states))
    backpointer = np.zeros((n_obs, n_states), dtype=int)
    
    # Initialize
    viterbi[0, :] = log_emission[0, :]
    
    # Forward pass
    for t in range(1, n_obs):
        for j in range(n_states):
            temp = viterbi[t-1, :] + log_transition[:, j]
            backpointer[t, j] = np.argmax(temp)
            viterbi[t, j] = np.max(temp) + log_emission[t, j]
    
    # Backtrack
    path = np.zeros(n_obs, dtype=int)
    path[-1] = np.argmax(viterbi[-1, :])
    for t in range(n_obs - 2, -1, -1):
        path[t] = backpointer[t+1, path[t+1]]
    
    return path.astype(int)

print("Post-processing functions defined")

## การฝึกอบรม Ensemble (Ensemble Training)

In [ ]:
@dataclass
class TrainResult:
    """ผลลัพธ์การฝึกอบรม"""
    y_pred_raw: np.ndarray
    y_pred_median: np.ndarray
    y_pred_viterbi: np.ndarray
    best_predictions: np.ndarray
    best_method: str
    y_true: np.ndarray
    models: List[Any]
    feature_names: List[str]

print("TrainResult dataclass defined")

In [ ]:
def train_ensemble(
    X_train: pd.DataFrame,
    y_train: np.ndarray,
    groups: np.ndarray
) -> TrainResult:
    """
    ฝึก ensemble: 4 LightGBM + 1 XGBoost กับ GroupKFold (5 folds)
    Evaluate OOF ด้วย raw, median filter, และ Viterbi
    """
    log("Starting ensemble training...")
    
    # 1. Select seed features
    seed_features = select_temporal_seed_features(X_train, y_train)
    
    # 2. Add temporal context
    X_temporal = add_temporal_context(X_train, seed_features)
    
    # 3. LightGBM configs
    lgb_configs = [
        {"n_estimators": 1500, "learning_rate": 0.025, "num_leaves": 127, "max_depth": -1, "subsample": 0.85, "colsample_bytree": 0.7, "reg_alpha": 0.3, "reg_lambda": 0.5, "min_child_samples": 25, "random_state": 42},
        {"n_estimators": 1200, "learning_rate": 0.035, "num_leaves": 63, "max_depth": 12, "subsample": 0.9, "colsample_bytree": 0.75, "reg_alpha": 0.1, "reg_lambda": 0.3, "min_child_samples": 40, "random_state": 314},
        {"n_estimators": 1000, "learning_rate": 0.05, "num_leaves": 255, "max_depth": -1, "subsample": 0.8, "colsample_bytree": 0.65, "reg_alpha": 0.5, "reg_lambda": 0.8, "min_child_samples": 20, "random_state": 777},
        {"n_estimators": 800, "learning_rate": 0.06, "num_leaves": 95, "max_depth": 8, "subsample": 0.85, "colsample_bytree": 0.8, "reg_alpha": 0.0, "reg_lambda": 0.1, "min_child_samples": 50, "random_state": 1234},
    ]
    xgb_config = {"n_estimators": 1000, "learning_rate": 0.03, "max_depth": 8, "subsample": 0.85, "colsample_bytree": 0.7, "reg_alpha": 0.3, "reg_lambda": 0.5, "min_child_weight": 25, "random_state": 999, "tree_method": "hist"}
    
    # 4. GroupKFold (5 folds)
    gkf = GroupKFold(n_splits=5)
    
    # OOF predictions
    oof_predictions = np.zeros((len(X_temporal), N_CLASSES))
    fold_idx = 0
    models_trained = []
    
    for train_indices, val_indices in gkf.split(X_temporal, y_train, groups):
        fold_idx += 1
        log(f"Fold {fold_idx}")
        
        X_train_fold = X_temporal.iloc[train_indices]
        X_val_fold = X_temporal.iloc[val_indices]
        y_train_fold = y_train[train_indices]
        y_val_fold = y_train[val_indices]
        
        # LightGBM models
        for config_idx, config in enumerate(lgb_configs):
            log(f"  LGB config {config_idx + 1}/4")
            model = lgb.LGBMClassifier(**config)
            model.fit(
                X_train_fold, y_train_fold,
                eval_set=[(X_val_fold, y_val_fold)],
                callbacks=[
                    lgb.early_stopping(120),
                    lgb.log_evaluation(period=0)
                ]
            )
            
            pred = model.predict_proba(X_val_fold)
            oof_predictions[val_indices] += pred / 5  # 5 models per fold
            models_trained.append(model)
        
        # XGBoost model
        log(f"  XGBoost")
        
        # Compute class weights
        from collections import Counter
        class_counts = Counter(y_train_fold)
        max_count = max(class_counts.values())
        class_weights = {cls: max_count / count for cls, count in class_counts.items()}
        sample_weights = np.array([class_weights[y] for y in y_train_fold])
        
        xgb_config_fold = xgb_config.copy()
        xgb_config_fold['random_state'] = xgb_config['random_state'] + 17 * (fold_idx - 1)
        
        model_xgb = xgb.XGBClassifier(**xgb_config_fold, eval_metric='mlogloss')
        model_xgb.fit(
            X_train_fold, y_train_fold,
            sample_weight=sample_weights,
            eval_set=[(X_val_fold, y_val_fold)],
            callbacks=[
                xgb.callback.EarlyStopping(rounds=120)
            ],
            verbose=False
        )
        
        pred = model_xgb.predict_proba(X_val_fold)
        oof_predictions[val_indices] += pred / 5
        models_trained.append(model_xgb)
    
    # Get class predictions from probabilities
    y_pred_raw = np.argmax(oof_predictions, axis=1)
    
    # 5. Post-processing
    y_pred_median = median_filter_smooth(y_pred_raw, window_size=5)
    y_pred_viterbi = viterbi_smooth(y_pred_raw, y_train)
    
    # 6. Evaluate and select best method
    acc_raw = accuracy_score(y_train, y_pred_raw)
    acc_median = accuracy_score(y_train, y_pred_median)
    acc_viterbi = accuracy_score(y_train, y_pred_viterbi)
    
    log(f"OOF Accuracy - Raw: {acc_raw:.4f}, Median: {acc_median:.4f}, Viterbi: {acc_viterbi:.4f}")
    
    results = {
        'raw': (y_pred_raw, acc_raw),
        'median': (y_pred_median, acc_median),
        'viterbi': (y_pred_viterbi, acc_viterbi)
    }
    best_method, (best_preds, best_acc) = max(results.items(), key=lambda x: x[1][1])
    log(f"Best method: {best_method} (acc={best_acc:.4f})")
    
    return TrainResult(
        y_pred_raw=y_pred_raw,
        y_pred_median=y_pred_median,
        y_pred_viterbi=y_pred_viterbi,
        best_predictions=best_preds,
        best_method=best_method,
        y_true=y_train,
        models=models_trained,
        feature_names=X_temporal.columns.tolist()
    )

print("train_ensemble function defined")

## การทำงาน (Execution)

In [ ]:
# ตั้งค่า paths
train_dir = DATA_DIR / "train" / "train"
test_dir = DATA_DIR / "test_segment" / "test_segment"
sample_sub_path = DATA_DIR / "sample_submission.csv"

print(f"Train dir: {train_dir}")
print(f"Test dir: {test_dir}")
print(f"Sample submission: {sample_sub_path}")

# Verify paths exist
print(f"Train dir exists: {train_dir.exists()}")
print(f"Test dir exists: {test_dir.exists()}")
print(f"Sample submission exists: {sample_sub_path.exists()}")

In [ ]:
# โหลดข้อมูลการฝึกอบรม
log("Loading training data...")
X_train, y_train, groups = load_training_data(train_dir)

print(f"X_train shape: {X_train.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"Unique subjects: {len(np.unique(groups))}")
print(f"Class distribution: {np.bincount(y_train)}")

In [ ]:
# โหลดข้อมูลการทดสอบ
log("Loading test data...")
test_data_by_subject, test_ids_by_subject = load_test_data(test_dir)

print(f"Test subjects: {list(test_data_by_subject.keys())}")
for subj_id, df_test in test_data_by_subject.items():
    print(f"  {subj_id}: {df_test.shape}")

In [ ]:
# ฝึก ensemble
log("Training ensemble...")
train_result = train_ensemble(X_train, y_train, groups)

print(f"Training complete!")
print(f"Best method: {train_result.best_method}")

In [ ]:
# สร้าง submission predictions สำหรับ test data
log("Generating test predictions...")

test_predictions_by_subject = {}

for subject_id, X_test in test_data_by_subject.items():
    log(f"Predicting for {subject_id}...")
    
    # Align features with training data
    missing_features = set(train_result.feature_names) - set(X_test.columns)
    for feat in missing_features:
        X_test[feat] = 0
    
    # Ensure same column order
    X_test = X_test[train_result.feature_names]
    
    # Predict with ensemble
    oof_test = np.zeros((len(X_test), N_CLASSES))
    
    for model in train_result.models:
        if isinstance(model, (lgb.LGBMClassifier, xgb.XGBClassifier)):
            pred = model.predict_proba(X_test)
            oof_test += pred / len(train_result.models)
    
    # Get predictions
    y_pred = np.argmax(oof_test, axis=1)
    
    # Apply post-processing (same as train_result.best_method)
    if train_result.best_method == 'median':
        y_pred = median_filter_smooth(y_pred, window_size=5)
    elif train_result.best_method == 'viterbi':
        y_pred = viterbi_smooth(y_pred, y_train)
    
    test_predictions_by_subject[subject_id] = y_pred

print("Test predictions generated!")

In [ ]:
# สร้าง submission CSV
log("Creating submission CSV...")

sample_sub = pd.read_csv(sample_sub_path)
print(f"Sample submission shape: {sample_sub.shape}")
print(f"Sample submission columns: {sample_sub.columns.tolist()}")
print(f"First few rows:\n{sample_sub.head()}")

# Map predictions
submission = sample_sub.copy()
submission['sleep_stage'] = 'N2'  # default

for idx, row in sample_sub.iterrows():
    # Parse segment_id to get subject_id
    segment_id = row['segment_id']
    # Try to match subject
    for subject_id in test_predictions_by_subject.keys():
        if subject_id in segment_id:
            # Find segment index
            if subject_id in test_ids_by_subject:
                segment_list = test_ids_by_subject[subject_id]
                if segment_id in segment_list:
                    seg_idx = segment_list.index(segment_id)
                    pred_label_idx = test_predictions_by_subject[subject_id][seg_idx]
                    submission.loc[idx, 'sleep_stage'] = LABEL_DECODER.get(int(pred_label_idx), 'N2')
            break

print(f"Submission shape: {submission.shape}")
print(f"Submission stage distribution:\n{submission['sleep_stage'].value_counts()}")
print(f"First few rows:\n{submission.head()}")

In [ ]:
# บันทึก submission
output_path = Path('/tmp/sleep_data/submission.csv')
submission.to_csv(output_path, index=False)

log(f"Submission saved to {output_path}")
print(f"File size: {output_path.stat().st_size} bytes")

# Download
from google.colab import files
files.download(str(output_path))